# Machine learning: problems, evidence, and real data

**Lecture 9 · Notebook 00 · CMOR 438 / INDE 577**  
**Core:** 95 minutes · **Practice:** 40 minutes · **Extension:** 30+ minutes

Machine learning is not one algorithm. It is a family of ways to learn predictive, descriptive, generative, or decision behavior from data and feedback. We will survey its major branches through real datasets from medicine, chemistry, and image recognition.

The guiding question is:

> **What information is available during learning, what output is required, and what evidence could support its use?**

## How to use this notebook

1. Read each data card and problem statement before its code.
2. Predict the shape and meaning of each input and output.
3. Read every plot as evidence about a particular representation—not decoration.
4. Treat library calls as implementations of assumptions, not proof that the assumptions are right.
5. Run from top to bottom in the **Rice DSM** kernel.

The datasets are stored in the repository's read-only SQLite teaching database, so the core route works offline. The separate ingestion script records their scikit-learn source. Follow the source links before reusing them: a convenient database is not a substitute for provenance.

## Learning objectives

You will be able to:

- distinguish supervised, unsupervised, and reinforcement learning by feedback signal;
- distinguish regression, classification, clustering, and dimensionality reduction;
- explain where semi-supervised, self-supervised, active, online, transfer, deep, and generative learning fit;
- use `fit`, `predict`, `transform`, and `fit_predict` without confusing their meanings;
- identify the observational unit, representation, target/objective, evaluation evidence, and deployment action;
- distinguish prediction, description, generation, control, and causal inference; and
- record provenance, license, collection process, limitations, and ethical risks before modeling.

## Why this matters in industry

“Use a neural network” does not define a problem. Teams first need to state whether a system predicts a number, assigns a category, discovers structure, compresses a signal, generates an object, or chooses actions over time.

| Principal feedback during learning | Major branch | Typical output |
| --- | --- | --- |
| Features **and observed targets** | Supervised learning | number, category, probability |
| Features, **no supplied target** | Unsupervised learning | representation, cluster, density, anomaly score |
| State, action, and delayed reward | Reinforcement learning | policy or value |

These branches describe the principal learning signal. They are not exclusive product labels: a generative model may use self-supervised pretraining and reinforcement feedback; a semi-supervised method combines labeled and unlabeled observations.

## Historical lens: learning did not begin with deep networks

Several mathematical traditions converged into modern machine learning:

| Period | Contribution | Enduring question |
| --- | --- | --- |
| 1805–1809 | Legendre and Gauss developed least squares for inconsistent astronomical observations | How should noisy measurements determine unknown parameters? |
| 1930s–1950s | Statistical decision theory and pattern recognition connected data, loss, and action | Which rule minimizes expected loss under uncertainty? |
| 1950 | Turing discussed the possibility of constructing a learning machine | Can useful behavior be acquired rather than exhaustively programmed? |
| 1958–1959 | Rosenblatt's perceptron and Samuel's checkers program made adaptive computation concrete | How can experience change future predictions or play? |
| 1980s–present | Larger datasets, automatic differentiation, accelerators, and networked software expanded scale | How do we train, evaluate, deploy, and govern learned systems reliably? |

This is not a march in which every new method replaces the previous one. Least squares, nearest neighbors, trees, probabilistic models, and neural networks solve different problems and remain useful. Modern scale changes what can be represented and optimized; it does not eliminate sampling assumptions, objectives, or scientific judgment.

**Primary historical anchors:** Turing, *Computing Machinery and Intelligence* (1950); Rosenblatt, *The Perceptron* (1958); Samuel, *Some Studies in Machine Learning Using the Game of Checkers* (1959).

In [ ]:
from __future__ import annotations

# SQLite is part of Python's standard library; no database server is needed.
import sqlite3

# These libraries provide plotting, arrays/dataframes, and ML estimators.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

# Import only the estimators and utilities used in this survey.
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    mean_absolute_error,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Resolve repository data safely even when VS Code changes the kernel's folder.
from rice_dsm.paths import course_database_path

# A fixed seed makes the random split and simulation reproducible.
RANDOM_SEED = 438
rng = np.random.default_rng(RANDOM_SEED)

# Fail early if an older environment cannot support the demonstrated API.
assert tuple(int(part) for part in sklearn.__version__.split(".")[:2]) >= (1, 7)
print(f"scikit-learn {sklearn.__version__}; seed={RANDOM_SEED}")

## A problem-first map

Before choosing an estimator, write:

1. **Unit and population:** what is one row, image, sequence, or episode; which future cases matter?
2. **Available information:** what can legitimately be known when the output is needed?
3. **Learning signal:** target, absence of target, reward, or a combination.
4. **Output and action:** number, label, representation, grouping, generated object, or policy—and who consumes it.
5. **Evidence:** held-out targets, stability, reconstruction behavior, intervention return, expert review, or another justified criterion.
6. **Data governance:** origin, consent/authority, license, sensitive fields, collection bias, and allowed uses.

The same data can support multiple mathematical tasks, but not every claim. Prediction from observational data is not automatically causal evidence about an intervention.

## The mathematical anatomy of a learning problem

In supervised learning, let

- $n$ be the number of observed units;
- $x_i\in\mathcal X$ be the feature representation of unit $i$;
- $y_i\in\mathcal Y$ be its observed target; and
- $D=\{(x_i,y_i)\}_{i=1}^n$ be the training sample.

Let $\mathcal{F}$ be a **hypothesis class**: the collection of functions the model is allowed to consider. A learning algorithm is itself a map

$$
\mathcal{A}:D\longmapsto \widehat f_D\in\mathcal{F}.
$$

Here $\mathcal A$ is the algorithm and $\widehat f_D$ is the function fitted from the particular sample $D$. The hat emphasizes estimation: another sample can produce another fitted function. Before calling `fit`, locate six choices:

1. **Representation** $\phi$: raw record $r\mapsto x=\phi(r)$.
2. **Hypothesis class** $\mathcal{F}$: which behaviors can be represented?
3. **Objective**: what counts as a better fit, representation, sample, or action?
4. **Algorithm** $\mathcal{A}$: how is one candidate selected or updated?
5. **Evaluation distribution**: to which future cases should evidence transfer?
6. **Decision path**: what person or system consumes the output?

For supervised learning, a loss function $L(y,\widehat y)$ assigns a numerical cost to predicting $\widehat y$ when the observed target is $y$. The **empirical risk** is the sample-average loss

$$
\widehat R_D(f)=\frac1n\sum_{i=1}^n L\bigl(y_i,f(x_i)\bigr).
$$

The learner searches for a function with small empirical risk, perhaps plus a regularization penalty. Evaluation asks a different question: does the fitted function have small loss on relevant **new** units? This distinction is why training performance is not sufficient evidence.

For KMeans, the objective instead measures squared distance to assigned centers. For PCA, it can be expressed through reconstruction error or retained variance. The word “learning” does not imply one universal loss.

Finite observations cannot determine behavior everywhere. The representation, function class, loss, regularization, and optimizer encode **inductive bias**—the preferences that make one continuation beyond the data more likely than another. There is no assumption-free learner.

## Worked examples: real-data gallery and provenance

We will query four real datasets from `data/course_datasets.sqlite`. The database was built from datasets distributed with the locked scikit-learn version:

| Domain | Dataset | Unit | Features | Example task |
| --- | --- | --- | --- | --- |
| Clinical research | Diabetes | patient | 10 baseline variables | regression |
| Medical imaging | Wisconsin Diagnostic Breast Cancer | sampled breast mass | 30 image-derived measurements | classification |
| Analytical chemistry | Wine recognition | wine sample | 13 chemical measurements | clustering |
| Computer vision | Optical handwritten digits | 8×8 image | 64 pixel intensities | dimensionality reduction |

These are real observations, but they are small historical teaching/benchmark datasets. None is automatically representative of a modern deployment population. The breast-cancer example is an algorithm demonstration, **not medical advice or a diagnostic device**.

The first query reads the catalog rather than guessing table meanings. Notice the read-only URI: analysis code should not silently mutate the shared source database.

### Paths belong to processes, not notebooks

A relative path such as `Path("data/course_datasets.sqlite")` is interpreted from the kernel process's **current working directory**. VS Code may start that process at the repository root, the notebook directory, or another configured location; the notebook file's location does not determine the path. The package helper below locates the repository deliberately and returns an absolute path. `Path.as_uri()` then creates a cross-platform SQLite URI, and `mode=ro` prevents accidental writes to the shared teaching database.

In [ ]:
# Resolve one absolute path, then ask SQLite to enforce read-only access.
database_path = course_database_path()
database_uri = f"{database_path.as_uri()}?mode=ro"
course_database = sqlite3.connect(database_uri, uri=True)

# Read metadata first: row meaning and provenance precede model fitting.
dataset_summary = pd.read_sql_query(
    """
    SELECT dataset_id, domain, task, observational_unit,
           row_count AS observations, feature_count AS features, source_url
    FROM dataset_catalog
    ORDER BY dataset_id
    """,
    course_database,
)

# This assertion is an executable expectation about the teaching snapshot.
assert dataset_summary["observations"].min() > 100
dataset_summary

## Inspect the database before selecting a model

A database is not a mysterious file that must already be understood. We can ask it what tables it contains, inspect the course data dictionary, preview rows, and then calculate diagnostics. This is **data discovery**, and it should precede modeling.

We will answer five questions in order:

1. **What relations exist?** Query `sqlite_master` so we do not guess table names.
2. **What does each column mean?** Query `dataset_columns` for roles and ordering. A feature, identifier, and target must not be treated interchangeably; using the target as an input creates **target leakage**.
3. **What does one row look like?** Preview a few records to catch parsing, units, and encoding surprises. A preview is not a statistical summary.
4. **Are values missing, duplicated, or implausible?** Missingness changes the usable population; duplicate units can invalidate a split; impossible ranges often reveal ingestion errors.
5. **What are the distributions and relationships?** Center, spread, skew, outliers, imbalance, and scale influence baselines, transformations, metrics, and model assumptions.

These checks do not certify a dataset. They make its structure visible enough to formulate better questions. Domain documentation and collection knowledge remain essential.

In [ ]:
# 1. Discover tables from SQLite's own catalog instead of guessing names.
available_tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    course_database,
)
display(available_tables)

# 2. Read semantic roles recorded during ingestion. This prevents target leakage.
diabetes_dictionary = pd.read_sql_query(
    """
    SELECT ordinal_position, column_name, role
    FROM dataset_columns
    WHERE dataset_id = ?
    ORDER BY ordinal_position
    """,
    course_database,
    params=("diabetes",),
)
display(diabetes_dictionary)

# 3. Load one row per patient only after confirming the table and its roles.
diabetes = pd.read_sql_query(
    """
    SELECT observation_id, age, sex, bmi, bp, s1, s2, s3, s4, s5, s6,
           disease_progression
    FROM diabetes_observations
    ORDER BY observation_id
    """,
    course_database,
)
display(diabetes.head())

# 4. Build a compact profile: dtype, missingness, uniqueness, and quantiles.
diabetes_profile = diabetes.describe().T
diabetes_profile.insert(0, "dtype", diabetes.dtypes.astype(str))
diabetes_profile.insert(1, "missing", diabetes.isna().sum())
diabetes_profile.insert(2, "unique", diabetes.nunique())
display(diabetes_profile)

# IDs should be unique; this protects the meaning of an independent patient row.
assert diabetes["observation_id"].is_unique
assert diabetes.isna().sum().sum() == 0


### Why visualize before fitting?

A summary table compresses information and can hide shape. Histograms reveal skew, gaps, concentration, and possible outliers. A feature–target scatterplot reveals whether a simple relationship is plausible, whether variance changes across the feature range, and whether a small number of observations dominate the visible pattern.

We inspect the target to understand the prediction scale, but from this point onward target-guided exploration must respect the training boundary. Repeatedly using held-out targets to choose features or models would turn the test set into training information.

In [ ]:
# Each panel answers a named question; none is included merely for decoration.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Feature histogram: where do we have support, and are tails or gaps visible?
axes[0].hist(diabetes["bmi"], bins=20, edgecolor="white")
axes[0].axvline(diabetes["bmi"].median(), color="crimson", linestyle="--")
axes[0].set(
    xlabel="baseline BMI (kg/m²)",
    ylabel="patients",
    title="Feature support and shape",
)

# Target histogram: what prediction scale and baseline variability matter?
axes[1].hist(
    diabetes["disease_progression"], bins=20, edgecolor="white"
)
axes[1].axvline(
    diabetes["disease_progression"].mean(),
    color="crimson",
    linestyle="--",
)
axes[1].set(
    xlabel="one-year progression score",
    ylabel="patients",
    title="Target scale and shape",
)

# Scatterplot: is an association visible, and does spread change with BMI?
axes[2].scatter(
    diabetes["bmi"], diabetes["disease_progression"], alpha=0.55
)
axes[2].set(
    xlabel="baseline BMI (kg/m²)",
    ylabel="one-year progression score",
    title="Possible association and spread",
)

plt.tight_layout()
plt.show()

## 1. Supervised regression: predict a quantitative target

The diabetes data contain ten baseline variables and a quantitative measure of disease progression one year later. Regression learns from pairs $(\boldsymbol x_i,y_i)$ with numeric $y_i$. In this deliberately small example, $x_i$ is patient $i$'s baseline BMI and $y_i$ is that patient's one-year progression score. We fit the affine function

$$
\widehat f(x)=\widehat\beta_0+\widehat\beta_1x,
$$

where $\widehat\beta_0$ is the fitted intercept and $\widehat\beta_1$ is the fitted slope. For a held-out patient, $\widehat y_i=\widehat f(x_i)$ is the prediction and $e_i=y_i-\widehat y_i$ is the residual. We summarize prediction error with

$$
\operatorname{MAE}=\frac{1}{m}\sum_{i=1}^{m}|y_i-\widehat y_i|,
$$

where $m$ is the number of held-out patients. We use one feature so the geometry is visible, not because BMI alone is a sufficient clinical model. The fitted line estimates an association useful for prediction under assumptions; it does not prove that changing BMI would cause the predicted change.

In [ ]:
# 1. Reuse the dataframe we already inspected rather than hiding data access.
assert diabetes.shape == (442, 12)

# 2. Hide 25% before fitting so evaluation uses unseen patients.
diabetes_train, diabetes_test = train_test_split(
    diabetes, test_size=0.25, random_state=RANDOM_SEED
)

# 3. Fit beta_0 and beta_1 using only the training patients.
diabetes_model = LinearRegression().fit(
    diabetes_train[["bmi"]], diabetes_train["disease_progression"]
)

# 4. Freeze the fitted model, predict the held-out targets, and score once.
diabetes_prediction = diabetes_model.predict(diabetes_test[["bmi"]])
diabetes_mae = mean_absolute_error(
    diabetes_test["disease_progression"], diabetes_prediction
)

# 5. Left: show the learned function and several training residuals.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(
    diabetes_train["bmi"],
    diabetes_train["disease_progression"],
    alpha=0.55,
    label="training patients",
)
bmi_domain = np.linspace(
    diabetes_train["bmi"].min(), diabetes_train["bmi"].max(), 120
)
fitted_line = diabetes_model.predict(pd.DataFrame({"bmi": bmi_domain}))
axes[0].plot(bmi_domain, fitted_line, color="crimson", label="fitted line")
residual_examples = diabetes_train.iloc[:10]
residual_fits = diabetes_model.predict(residual_examples[["bmi"]])
axes[0].vlines(
    residual_examples["bmi"],
    residual_fits,
    residual_examples["disease_progression"],
    color="gray",
    alpha=0.55,
    label="example residuals",
)
axes[0].set(
    xlabel="baseline BMI (kg/m²)",
    ylabel="one-year progression score",
    title="Fit uses training patients only",
)
axes[0].legend()

# Right: distance from the diagonal is held-out prediction error.
axes[1].scatter(
    diabetes_test["disease_progression"], diabetes_prediction, alpha=0.65
)
limits = [
    diabetes_test["disease_progression"].min(),
    diabetes_test["disease_progression"].max(),
]
axes[1].plot(limits, limits, color="black", linestyle="--")
axes[1].set(
    xlabel="observed score $y$",
    ylabel="predicted score $\widehat y$",
    title=f"Held-out MAE = {diabetes_mae:.1f} points",
)
plt.tight_layout()
plt.show()

# Executable checks catch accidental changes in shape or model behavior.
assert len(diabetes_prediction) == len(diabetes_test)
assert diabetes_mae < 70

## 2. Supervised classification: predict a category

The Wisconsin Diagnostic Breast Cancer data contain features computed from digitized images of fine-needle aspirates. Let $x_i\in\mathbb R^{30}$ be the vector of image-derived measurements and let $y_i\in\{0,1\}$ encode malignant or benign. Logistic regression estimates a conditional probability

$$
\widehat p(x)=P(Y=1\mid X=x),
$$

then converts that probability to a class using a decision threshold. A different threshold changes the tradeoff between the two kinds of error; it does not change the underlying fitted probabilities.

We fit all 30 features but separately plot two measurements to show overlap. In the confusion matrix, rows are observed classes and columns are predicted classes. Accuracy and a confusion matrix demonstrate interfaces only; clinical use would demand carefully defined sensitivity/specificity costs, probability calibration, external validation, subgroup audits, workflow study, and regulatory evidence.

In [ ]:
# 1. Load one row per sampled breast mass.
cancer = pd.read_sql_query(
    "SELECT * FROM breast_cancer_observations ORDER BY observation_id",
    course_database,
)

# 2. Preserve the class proportions in both partitions with stratification.
cancer_train, cancer_test = train_test_split(
    cancer,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=cancer["diagnosis_code"],
)

# Identifiers and targets describe rows; they must not become predictors.
cancer_features = [
    column for column in cancer.columns
    if column not in {"observation_id", "diagnosis_code", "diagnosis_label"}
]

# 3. Fit scaling inside the pipeline to prevent test-data leakage.
cancer_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
cancer_model.fit(cancer_train[cancer_features], cancer_train["diagnosis_code"])

# 4. Keep both outputs: probabilities express uncertainty; labels encode a decision.
cancer_probability_benign = cancer_model.predict_proba(
    cancer_test[cancer_features]
)[:, 1]
cancer_prediction = cancer_model.predict(cancer_test[cancer_features])
cancer_accuracy = accuracy_score(cancer_test["diagnosis_code"], cancer_prediction)

# 5. Each panel answers a different question about the fitted system.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
cancer_target_names = np.array(["malignant", "benign"])
for class_value, class_name in enumerate(cancer_target_names):
    subset = cancer_train[cancer_train["diagnosis_code"] == class_value]
    axes[0].scatter(
        subset["mean_radius"],
        subset["mean_texture"],
        alpha=0.5,
        label=class_name,
    )
axes[0].set(
    xlabel="mean radius",
    ylabel="mean texture",
    title="Two features overlap",
)
axes[0].legend()

# Probability distributions reveal confidence that hard labels hide.
for class_value, class_name in enumerate(cancer_target_names):
    class_mask = cancer_test["diagnosis_code"].to_numpy() == class_value
    axes[1].hist(
        cancer_probability_benign[class_mask],
        bins=np.linspace(0, 1, 16),
        alpha=0.6,
        label=f"observed {class_name}",
    )
axes[1].axvline(0.5, color="black", linestyle="--", label="default threshold")
axes[1].set(
    xlabel="estimated $P(Y=1\;[\mathrm{benign}]\mid X=x)$",
    ylabel="held-out masses",
    title="Probability precedes decision",
)
axes[1].legend(fontsize=8)

# The confusion matrix counts the four observed/predicted combinations.
ConfusionMatrixDisplay.from_predictions(
    cancer_test["diagnosis_code"], cancer_prediction,
    display_labels=cancer_target_names, colorbar=False, ax=axes[2],
)
axes[2].set_title(f"Held-out accuracy = {cancer_accuracy:.3f}")
plt.tight_layout()
plt.show()

# Probabilities and labels must align one-for-one with the held-out rows.
assert cancer_probability_benign.shape == cancer_prediction.shape
assert np.all((cancer_probability_benign >= 0) & (cancer_probability_benign <= 1))
assert set(cancer_prediction) <= {0, 1}
assert cancer_accuracy > 0.90

## 3. Unsupervised clustering: search for groups without labels

The wine data contain 13 chemical measurements from 178 samples and known cultivar labels. `KMeans` will **not** receive those labels. Let $x_i\in\mathbb R^{13}$ be the scaled chemical measurements for sample $i$, let $z_i\in\{1,\ldots,K\}$ be its assigned cluster, and let $\mu_k$ be cluster $k$'s center. KMeans approximately minimizes

$$
J(z,\mu)=\sum_{i=1}^{n}\left\lVert x_i-\mu_{z_i}\right\rVert_2^2.
$$

We standardize each feature because squared Euclidean distance is sensitive to units: an abundant chemical measured on a large numerical scale could otherwise dominate. We use the known cultivar only afterward as an audit aid. Agreement is neither guaranteed nor the definition of successful clustering: scientific structure may differ from the recorded label, and preprocessing plus distance metric partly define what “similar” means. PCA supplies only the two plotting coordinates; KMeans fits in all 13 scaled dimensions.

In [ ]:
# 1. Load the chemical measurements and keep labels outside the feature matrix.
wine = pd.read_sql_query(
    "SELECT * FROM wine_observations ORDER BY observation_id", course_database
)
wine_feature_names = [
    column for column in wine.columns
    if column not in {"observation_id", "cultivar_code", "cultivar_label"}
]
wine_features = wine[wine_feature_names]

# 2. Equalize feature scales before using Euclidean distance.
wine_scaled = StandardScaler().fit_transform(wine_features)

# 3. Fit KMeans without cultivar labels; n_init reduces sensitivity to starts.
wine_clusterer = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=20)
wine_clusters = wine_clusterer.fit_predict(wine_scaled)

# 4. Compress to two coordinates for plotting only, not for cluster fitting.
wine_coordinates = PCA(
    n_components=2, random_state=RANDOM_SEED
).fit_transform(wine_scaled)

# 5. Compare two colorings of the same coordinates after fitting.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True)
first = axes[0].scatter(
    wine_coordinates[:, 0],
    wine_coordinates[:, 1],
    c=wine_clusters,
    cmap="viridis",
    s=38,
)
axes[0].set(
    title="KMeans groups (labels hidden)",
    xlabel="principal component 1",
    ylabel="principal component 2",
)
second = axes[1].scatter(
    wine_coordinates[:, 0],
    wine_coordinates[:, 1],
    c=wine["cultivar_code"],
    cmap="viridis",
    s=38,
)
axes[1].set(
    title="Known cultivars (audit only)", xlabel="principal component 1"
)
fig.colorbar(first, ax=axes[0], label="cluster ID")
fig.colorbar(second, ax=axes[1], label="cultivar ID")
plt.tight_layout()
plt.show()

# Cluster IDs are arbitrary names; only their partition has meaning.
assert wine_clusters.shape == (178,)
assert np.unique(wine_clusters).size == 3
assert wine_clusterer.inertia_ > 0

## 4. Dimensionality reduction: learn a representation

Each handwritten digit is an $8\times8$ image represented by 64 pixel intensities. After centering and scaling, write the data matrix as $X\in\mathbb R^{n\times64}$. PCA chooses a unit direction $v_1$ whose projected coordinates have maximum sample variance,

$$
v_1=\arg\max_{\lVert v\rVert_2=1}\operatorname{Var}(Xv),
$$

then chooses orthogonal directions $v_2,v_3,\ldots$. The coordinates $z_{ij}=x_i^Tv_j$ are a new representation of observation $i$. The **explained-variance ratio** reports the fraction of total variance associated with each direction. PCA receives no digit labels; labels color the final plot only so humans can interpret the representation.

A two-dimensional projection is useful for seeing broad structure but cannot preserve every distance or class boundary from 64 dimensions. The variance bar chart makes that information loss explicit. Apparent separation in a plot is exploratory evidence, not held-out classification performance.

In [ ]:
# 1. Load 64 pixel columns and restore each flattened row to an 8x8 image.
digits = pd.read_sql_query(
    "SELECT * FROM digits_observations ORDER BY observation_id", course_database
)
digit_feature_names = [f"pixel_{index}" for index in range(64)]
digit_features = digits[digit_feature_names].to_numpy()
digit_images = digit_features.reshape(-1, 8, 8)

# 2. Inspect raw observations before reducing their dimension.
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for ax, image, label in zip(
    axes.flat, digit_images[:12], digits["digit_label"][:12], strict=True
):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"label {label}")
    ax.axis("off")
fig.suptitle("Real 8×8 handwritten-digit images")
plt.tight_layout()
plt.show()

# 3. Standardize pixels so variance comparisons are on a common scale.
digits_scaled = StandardScaler().fit_transform(digit_features)

# 4. Fit PCA without labels and retain two coordinates for visualization.
digits_pca = PCA(n_components=2, random_state=RANDOM_SEED)
digit_coordinates = digits_pca.fit_transform(digits_scaled)

# 5. Show both the representation and how much variation two axes retain.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter = axes[0].scatter(
    digit_coordinates[:, 0],
    digit_coordinates[:, 1],
    c=digits["digit_label"],
    cmap="tab10",
    s=12,
    alpha=0.7,
)
axes[0].set(
    xlabel="principal component 1",
    ylabel="principal component 2",
    title="Labels interpret; they did not fit PCA",
)
fig.colorbar(scatter, ax=axes[0], ticks=range(10), label="digit label")

variance_percent = 100 * digits_pca.explained_variance_ratio_
axes[1].bar([1, 2], variance_percent)
axes[1].set(
    xticks=[1, 2],
    xlabel="principal component",
    ylabel="explained variance (%)",
    title=f"Two axes retain {variance_percent.sum():.1f}% of variance",
)
plt.tight_layout()
plt.show()

# Shape and variance checks make the compression contract explicit.
assert digit_coordinates.shape == (1797, 2)
assert 0 < digits_pca.explained_variance_ratio_.sum() < 1

## 5. Reinforcement learning: actions change later data

In reinforcement learning, an agent observes a state, selects an action, and receives a possibly delayed reward. The output is a policy. Unlike an ordinary fixed dataset, the policy changes which observations are collected.

A multi-armed bandit is the special case with no changing state and an immediate reward. At decision step $t$, the agent chooses $A_t\in\{0,1,2\}$ and observes binary reward $R_t\in\{0,1\}$. Each action has an unknown success probability $p_a=P(R_t=1\mid A_t=a)$. After selecting action $a$ a total of $N_t(a)$ times, its sample-mean value estimate is

$$
\widehat Q_t(a)=\frac{\sum_{s<t}R_s\,\mathbf 1(A_s=a)}{N_t(a)}.
$$

An $\varepsilon$-greedy policy chooses a random action with probability $\varepsilon$ (**exploration**) and otherwise chooses the largest current estimate (**exploitation**). A clean real-data demonstration requires a logged policy, action propensities, rewards, and assumptions for off-policy evaluation—or a safe interactive environment. We therefore use a **declared simulation** for mechanism, not a synthetic table disguised as empirical evidence.

In [ ]:
# Hidden environment parameters: the agent never receives this array directly.
true_success_probability = np.array([0.28, 0.47, 0.39])
epsilon = 0.10
number_of_steps = 500

# Sufficient statistics let us update each action's sample mean online.
action_count = np.zeros(3, dtype=int)
reward_sum = np.zeros(3, dtype=float)
reward_history = []
actions = []
exploration_count = 0

for _step in range(number_of_steps):
    # Estimate Q(a) safely; untried actions temporarily receive value zero.
    estimated_value = np.divide(
        reward_sum, action_count, out=np.zeros(3), where=action_count > 0
    )

    # Force every action to be tried, then explore with probability epsilon.
    explore = rng.random() < epsilon or np.any(action_count == 0)
    action = int(rng.integers(3) if explore else np.argmax(estimated_value))
    exploration_count += int(explore)

    # The environment samples a Bernoulli reward for the selected action.
    reward = float(rng.random() < true_success_probability[action])

    # Update only the selected action; this changes future choices.
    action_count[action] += 1
    reward_sum[action] += reward
    actions.append(action)
    reward_history.append(reward)

# Recompute final estimates and learning curves after interaction ends.
estimated_value = reward_sum / action_count
running_average_reward = np.cumsum(reward_history) / np.arange(1, number_of_steps + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(range(3), action_count)
axes[0].set(
    xlabel="action",
    ylabel="times selected",
    title="Policy changes data collection",
)

action_axis = np.arange(3)
axes[1].bar(action_axis - 0.18, true_success_probability, width=0.36, label="true")
axes[1].bar(action_axis + 0.18, estimated_value, width=0.36, label="estimated")
axes[1].set(
    xticks=action_axis,
    xlabel="action",
    ylabel="success probability",
    title="Experience improves estimates",
)
axes[1].legend()

axes[2].plot(running_average_reward)
axes[2].axhline(
    true_success_probability.max(),
    color="black",
    linestyle="--",
    label="best fixed action",
)
axes[2].set(
    xlabel="decision step",
    ylabel="running average reward",
    title="Learning while earning reward",
)
axes[2].legend()
plt.tight_layout()
plt.show()

# The best action should dominate, while epsilon keeps alternatives observable.
assert sum(action_count) == number_of_steps
assert exploration_count > 0
assert action_count[np.argmax(true_success_probability)] == action_count.max()
course_database.close()

## Related paradigms are different axes

- **Semi-supervised learning:** a small labeled set plus a larger unlabeled set.
- **Self-supervised learning:** targets are constructed from the data, such as masked tokens or image regions.
- **Generative modeling:** learn a distribution or conditional distribution to create samples, text, images, molecules, or other objects.
- **Active learning:** choose which expensive label to request next.
- **Online learning:** update as observations arrive rather than in one fixed batch.
- **Transfer learning:** adapt representations or parameters learned on one task/domain to another.
- **Deep learning:** use multilayer parameterized architectures; it can appear in supervised, self-supervised, generative, or reinforcement settings.

Also distinguish **causal inference**. Predictive association asks what output is likely given observed features. A causal claim asks what would change under an intervention and needs additional design or assumptions.

## Application gallery: the branch does not define the whole project

| Domain | Possible task | Output | Evidence that would matter |
| --- | --- | --- | --- |
| Astronomy | Classify transient events from image and light-curve features | class probabilities | later spectroscopic labels, time/site shift, calibration |
| Materials | Predict later battery capacity from early cycles | capacity with uncertainty | unseen cells, later batches, new chemistries |
| Ecology | Estimate species occurrence from surveys and remote sensing | occurrence probability | spatial holdout, detection process, field validation |
| Manufacturing | Detect unusual sensor trajectories | anomaly score | maintenance outcomes, equipment generations, false-alarm burden |
| Chemistry | Propose molecules satisfying property constraints | ranked/generated candidates | validity, novelty, synthesizability, laboratory confirmation |
| Operations | Choose inventory under uncertain demand | decision policy | realized cost under temporal evaluation and constraints |

An algorithm demonstration can show an interface. It cannot by itself establish representativeness, usefulness, safety, or deployment readiness. Each row needs a different observational unit, split, metric, and feedback path.

## Common failure modes

- **Benchmark blindness:** a convenient historical dataset is treated as representative of current deployment.
- **Label mythology:** a target is assumed to be objective truth without examining how and why it was constructed.
- **Unit leakage:** one patient, device, site, author, or time period appears on both sides of evaluation.
- **Unsupervised overclaiming:** visually pleasing clusters are called natural kinds without stability or domain evidence.
- **Metric substitution:** an easy score replaces the actual decision cost.
- **License/provenance loss:** a CSV is copied without origin, version, authority, or permitted use.
- **Causal language:** a predictive coefficient or feature importance is presented as an intervention effect.

## Debugging and data triage

1. Print shapes, dtypes, target values, missingness, and duplicated units.
2. Read the data card, original source, and license before interpreting column names.
3. Plot raw distributions and representative observations—not only a model score.
4. Split the unit that must generalize before target-guided analysis.
5. Fit transformations inside the training boundary.
6. Compare with a trivial baseline and inspect errors by meaningful slice.
7. Ask whether the dataset contains the metadata needed to evaluate the intended claim.

## Professional practice

Every project should keep a small data-source record:

- stable landing page and dataset identifier;
- creator, steward, citation, and date/version accessed;
- license or usage terms;
- collection and sampling process;
- meaning of one observation and every target;
- units, valid ranges, missing-value codes, and transformations;
- sensitive attributes and foreseeable harms;
- known limitations and prohibited claims; and
- checksum or immutable snapshot reference when redistribution is permitted.

A model registry without a data lineage record is incomplete.

## Guided practice: inspect, then formulate

Choose the digits or wine dataset. First perform a short data triage:

1. discover its observation and data-dictionary tables with SQL;
2. preview five rows and state what one row represents;
3. report shape, dtypes, missingness, uniqueness, and descriptive statistics;
4. draw one distribution plot and one relationship or representative-observation plot; and
5. write one sentence explaining what each diagnostic could reveal and what it cannot establish.

Then write two different ML problems that use the same rows. For each, state unit, features available at prediction time, learning signal, output, action, and evaluation evidence.

**Success criteria:** inspection code must be separated from model fitting; every plot must answer a named question; the two tasks must make different claims; one task must be unsupervised; and neither may use a label while claiming it was unavailable.

In [ ]:
# A mechanical provenance checkpoint to extend in your own work.
provenance_record = {
    "name": "Optical recognition of handwritten digits",
    "table": "digits_observations",
    "source": "UCI ML handwritten digits test set",
    "observational_unit": "one 8x8 processed handwritten-digit image",
    "features": 64,
    "target": "digit identity 0-9 (not used by PCA fitting)",
}
assert provenance_record["features"] == digit_features.shape[1]
provenance_record

## Independent practice

Select a new real dataset from one of the repositories below. Create a one-page data card and three visualizations before fitting a model: distribution of the proposed target, distribution/support of key features, and one plot that could reveal selection, missingness, time, group, or class imbalance.

**Success criteria:** link the primary landing page; record license and version/date; define the row and target; identify at least one unsupported claim; and keep the raw download out of Git if redistribution terms or size make that inappropriate.

## Extension: compare domains without comparing scores

Choose two datasets from different domains. Explain why their metric values cannot be compared as if they measured task difficulty. Examine sample construction, target ambiguity, feature acquisition cost, class/target distribution, and consequences of error.

## Where to find public data

Start with sources that preserve metadata and a stable landing page:

- [UCI Machine Learning Repository](https://archive.ics.uci.edu/) — curated ML datasets across many domains.
- [OpenML](https://www.openml.org/) — versioned datasets, tasks, and benchmark runs.
- [Data.gov](https://data.gov/) — United States government open-data catalog.
- [NASA Science Data](https://science.data.nasa.gov/) and [NOAA Data Discovery](https://data.noaa.gov/onestop/) — space, Earth, weather, ocean, and climate data.
- [NIH data repositories](https://sharing.nih.gov/data-management-and-sharing-policy/sharing-scientific-data/repositories-for-sharing-scientific-data) — repository guidance for biomedical data.
- [World Bank Open Data](https://data.worldbank.org/) — global development indicators.
- [Google Dataset Search](https://datasetsearch.research.google.com/) — a search engine; verify the underlying steward and license yourself.

“Publicly reachable” does not necessarily mean openly licensed, ethically appropriate, representative, clean, or safe to redistribute.

## Retrieval practice

1. How can you discover a SQLite database's tables without already knowing their names?
2. Why are `head()` and `describe()` useful but insufficient in different ways?
3. What distinguishes the three major branches by feedback signal?
4. How do regression and classification differ mathematically?
5. Why can known wine labels be used to audit but not fit the clustering example?
6. What information is lost in a two-component PCA plot?
7. What do $N_t(a)$ and $\widehat Q_t(a)$ mean in the bandit example?
8. Why is a simulation appropriate for the bandit mechanism but not empirical evidence about real instruments?
9. Why is prediction not automatically causal inference?
10. Which provenance fields must travel with a dataset?

## Takeaway

Start with the observational unit, learning signal, output, action, and evidence—not an algorithm name. Real data make domain assumptions and limitations visible, but “real” does not imply representative, ethical, or deployment-ready. The mathematical branch and the data lineage jointly determine what can be claimed.

**Next:** the supervised-learning notebook derives ordinary least squares and then builds a complete held-out regression pipeline using the real diabetes data.

## Further reading

- [scikit-learn: toy dataset descriptions and original sources](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [scikit-learn: getting started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn: common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [NumPy array fundamentals](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Python tutorial: data structures](https://docs.python.org/3/tutorial/datastructures.html)